In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lingululwanishine/playground-series-s4e4/sample_submission.csv
/kaggle/input/datasets/lingululwanishine/playground-series-s4e4/test.csv
/kaggle/input/datasets/lingululwanishine/playground-series-s4e4/train.csv


In [2]:
from pathlib import Path
 
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
 
INPUT_DIR = Path("/kaggle/input/datasets/lingululwanishine/playground-series-s4e4")
OUTPUT_DIR = Path("/kaggle/working")
TARGET = "Rings"
ID = "id"
 
train = pd.read_csv(INPUT_DIR / "train.csv")
test = pd.read_csv(INPUT_DIR / "test.csv")
sample_submission = pd.read_csv(INPUT_DIR / "sample_submission.csv")
 
y_log = np.log1p(train[TARGET])
 
# Encode categorical features (including Sex) consistently across train and test.
X_train_raw = train.drop(columns=[TARGET, ID], errors="ignore")
X_test_raw = test.drop(columns=[ID], errors="ignore")
combined = pd.concat([X_train_raw, X_test_raw], axis=0, ignore_index=True)
 
categorical_cols = combined.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()
 
combined = pd.get_dummies(
    combined, columns=categorical_cols, dtype=float
)
combined = combined.apply(pd.to_numeric, errors="coerce")
combined = combined.fillna(combined.median()).fillna(0)
 
X = combined.iloc[:len(train)].reset_index(drop=True)
X_test = combined.iloc[len(train):].reset_index(drop=True)
 
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)
 
model = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42,
)
 
model.fit(X_train, y_train)
valid_log_preds = model.predict(X_valid)
rmsle = np.sqrt(mean_squared_error(y_valid, valid_log_preds))
print(f"Histogram Gradient Boosting validation RMSLE: {rmsle:.5f}")
 
# Refit on all training rows, then create the Kaggle submission.
model.fit(X, y_log)
test_log_preds = model.predict(X_test)
test_preds = np.maximum(np.expm1(test_log_preds), 0)
 
sample_submission[TARGET] = test_preds
output_path = OUTPUT_DIR / "submission_gradient_boosting.csv"
sample_submission.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Histogram Gradient Boosting validation RMSLE: 0.15181
Saved: /kaggle/working/submission_gradient_boosting.csv
